# 越南公开日线数据审计

## tl;dr

VCI/Vietcap 与 KBS 的公开日线接口可以支持低成本研究原型，但不能直接视为生产数据源。样本中存在交易所代码差异、`countBack` 日期语义、低流动性 OHLC 异常、返回顺序和价格单位等需要治理的问题。

## Context & Methods

本 notebook 配合 `exploration/01_public_ohlcv_probe.py` 使用。先运行脚本，再读取被 `.gitignore` 忽略的 `artifacts/public_ohlcv_probe.json`。审计只保留请求元数据和质量计数，不保存完整行情。

测试对象：VCI 证券列表、VCI 日线 OHLCV，以及 FPT 的 KBS 日线样本。质量规则包括必需字段缺失、重复时间戳、时间顺序、OHLC 关系和非负价格/成交量。

## Data

数据来源是公开 HTTP 接口；本 notebook 不内嵌原始市场数据。

In [1]:
from pathlib import Path
import json

artifact = Path('artifacts/public_ohlcv_probe.json')
if not artifact.exists():
    raise FileNotFoundError('先运行 python exploration/01_public_ohlcv_probe.py')
probe = json.loads(artifact.read_text(encoding='utf-8'))
print('retrieved_at_utc:', probe['retrieved_at_utc'])
print('listing_status:', probe['listing']['status_code'])
print('listing_rows:', probe['listing']['row_count'])
print('exchange_groups:', ', '.join(probe['listing']['sample_by_exchange']))
sample = [symbol for group in probe['listing']['sample_by_exchange'].values() for symbol in group[:2]]
print('VCI sample symbols:', ', '.join(sample))
print('FPT matched trading dates across VCI/KBS:', probe['cross_source']['FPT']['matched_trading_dates'])

retrieved_at_utc: 2026-08-26T16:53:29Z
listing_status: 200
listing_rows: 3586
exchange_groups: DELISTED, HNX, HOSE, UPCOM
VCI sample symbols: AAA, AAM, ADC, ALT, A32, AAH
FPT matched trading dates across VCI/KBS: 22


In [2]:
print('symbol  status  rows  missing  duplicate_time  invalid_ohlc  sorted')
for symbol, record in probe['ohlcv'].items():
    q = record['quality']
    print(f"{symbol:<7}{record['status_code']:<8}{q['row_count']:<6}{q['missing_required_cells']:<9}{q['duplicate_timestamps']:<17}{q['invalid_ohlc_rows']:<14}{q['sorted_non_decreasing']}")

symbol  status  rows  missing  duplicate_time  invalid_ohlc  sorted
AAA     200     1000  0        0                0             True
AAM     200     1000  0        0                0             True
ADC     200     1000  0        0                1             True
ALT     200     1000  0        0                0             True
A32     200     1000  0        0                65            True
AAH     200     15    0        0                0             True


## Results

VCI 的样本响应均为 HTTP 200，必需字段没有空值，时间戳没有重复。`A32` 和 `ADC` 触发了机械 OHLC 规则，优先应回看具体日期、零成交/停牌状态和原始字段。VCI 返回的 `HSX` 已在探针中规范为 `HOSE`。

VCI 的 `to + countBack` 请求返回的是截至结束点向前的若干根 K 线，不是严格的起止日期过滤；KBS 原始日线返回倒序。FPT 的单月交叉源比较匹配 22 个交易日，价格单位统一后最大收盘差约 0.01054 千 VND，最大相对差约 0.00015。

## Takeaways

1. 可以继续做日频价格/成交量/流动性因子原型。
2. 标准化层必须保留 `exchange_raw`、原始价格单位、请求参数、排序状态和严格日期裁剪。
3. 下一轮应扩大到 50 只股票、连续 5 个交易日，并接入 SSI/VSDC 样本；在此之前不把当前结果称为生产级或 point-in-time 数据。